# 📗 도구 셋으로 답하는 종합 에이전트

지난 시간에 우리는 도구를 **만들고**, 검색과 조회를 파이프라인으로 **엮었습니다.** 그리고 하나를 미뤄 두었습니다 — **체인은 무엇을 묻든 검색한다**는 문제였지요.

그 문제의 답은 "경로를 우리가 고정하지 않는 것"입니다. 무엇을 할지 **모델이 매번 스스로 정하게** 하는 것이죠. 이 사고 방식에 이름이 있습니다 — **ReAct**(Reasoning + Acting)입니다.

오늘은 그 위에 **하나의 창구**를 세웁니다. 사내에서 AI 서비스를 도입할 때 쏟아지는 질문은 **성격이 제각각**입니다.

| 들어오는 질문 | 답이 어디에 있나 | 오늘 만들 도구 |
|---|---|---|
| "대화를 학습에 쓰려면 무엇을 알려야 하나요?" | **문서**(안내서 74쪽) | 문서 검색 도구 |
| "국외로 데이터가 나가는 서비스가 몇 개죠?" | **표**(사내 도입 대장) | 표 조회 도구 |
| "검토가 언제 끝나나요?" | **바깥 세상**(공휴일·달력) | 외부 REST API 도구 |

셋 다 "검색"으로는 답할 수 없습니다. 그래서 오늘의 목표는 **셋을 각각 도구로 만들고, 무엇을 쓸지 모델이 고르게 하는 것**입니다.

## ⏪ 복습

| 언제 | 한 일 | 오늘 |
|---|---|---|
| 지난 단원 | 함수를 `@tool` 로 만들고 `create_agent` 에 붙였다 | **그 위에 ReAct 라는 이름을 얹는다** |
| 지난 단원 | 문서를 잘라 색인하고 RAG **체인**을 만들었다 | 그 검색을 **도구로 감싼다** |
| 지난 단원 | 자연어를 SQL 로 바꿔 조회했다 | 그 조회를 **같은 창구에 붙인다** |
| 지난 단원 | "체인은 인사말에도 검색이 돈다" 를 남겨 뒀다 | **오늘 실측으로 해결한다** |

> 오늘 쓰는 자료는 **실제 발간 문서**입니다 — 개인정보보호위원회가 배포한 「생성형 AI 개인정보 처리 안내서」와 「개인정보 처리방침 작성지침」 두 건(74쪽)입니다. 표는 그 규정을 우리 회사에 대입한 **사내 AI 서비스 도입 대장**(가상 데이터)입니다.

### 📚 오늘 배우는 것들의 공식 문서

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| 에이전트와 도구 선택 | [Agents](https://docs.langchain.com/oss/python/langchain/agents) |
| 도구 정의와 설명 | [Tools](https://docs.langchain.com/oss/python/langchain/tools) |
| 메시지 종류(`AIMessage`·`ToolMessage`) | [Messages](https://docs.langchain.com/oss/python/langchain/messages) |
| 실행 과정 관찰(`stream`) | [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) |
| 검색기·RAG | [Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval) |
| ReAct 원논문 | [ReAct: Synergizing Reasoning and Acting (arXiv)](https://arxiv.org/abs/2210.03629) |

**오늘의 목표**

- [ ] **ReAct 세 단계** — 생각·행동·관찰이 무엇이고, 메시지 기록의 어디에 남는지 짚는다.
- [ ] **외부 REST API 를 도구로** — 인터넷 서비스를 부르는 도구를 만들고, 실패까지 다룬다.
- [ ] **문서를 도구로** — 긴 문서를 조각내 검색하고, **앞뒤 조각까지 붙여** 문맥을 살리며, 찾은 내용에만 근거해 **쪽 번호와 함께** 답하게 한다.
- [ ] **표를 도구로** — 스키마 설명으로 SQL 을 만들게 하고, 가드를 두 겹으로 건다.
- [ ] **한 창구에 넷** — 라우팅·동시 호출·다단계 연쇄를 기록으로 확인한다.
- [ ] **선택이 틀어질 때** — 안 부름·잘못 고름·조용한 실패·과다 호출을 진단하고 고친다.

아래 준비 셀들을 먼저 실행하세요(임베딩 모델을 올리느라 처음 한 번은 잠시 걸립니다).

> **벡터DB 는 이미 만들어져 있습니다** — `data/chroma_day20/` 를 **열기만** 합니다. 74쪽을 조각내 임베딩하는 일은 커널을 켤 때마다 반복할 이유가 없어서 미리 만들어 함께 배포했습니다. **어떻게 만들었는지는 2절에서 코드로 보여 드립니다.**

> 색인이 **두 개**입니다. 데모는 **「생성형 AI 개인정보 처리 안내서」**(`policy_retriever`)로 하고, **함께 따라하기는 「개인정보 처리방침 작성지침」**(`std_retriever`)으로 합니다 — 같은 절차를 다른 문서에 직접 적용해 봐야 손에 남기 때문입니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델. 18일차에서 배운 그대로입니다(이 셀은 실행만 하세요).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
# [제공 코드] 미리 만들어 둔 벡터DB 를 엽니다 - 색인은 이미 끝나 있습니다(이 셀은 실행만 하세요).
#  임베딩 모델은 '질문을 벡터로 바꾸는 데' 필요해 한 번 올립니다(처음 한 번은 잠시 걸립니다).
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 적재할 때 쓴 것과 같은 모델이라야 순위가 맞습니다

policy_store = Chroma(persist_directory='data/chroma_day20',
                      collection_name='policy_day20',
                      embedding_function=embeddings)
policy_retriever = policy_store.as_retriever(search_kwargs={'k': 3})
print('생성형AI 안내서 조각 수:', len(policy_store.get()['ids']))

std_store = Chroma(persist_directory='data/chroma_day20',
                      collection_name='std_day20',
                      embedding_function=embeddings)
std_retriever = std_store.as_retriever(search_kwargs={'k': 3})
print('처리방침 작성지침 조각 수:', len(std_store.get()['ids']))

---
# 1. 생각하고 행동하고 관찰하는 ReAct

## 왜 필요할까요?
언어모델은 혼자서는 **최신 정보도, 정확한 계산도** 보장하지 못합니다. 그래서 **도구**를 손에 쥐여 줍니다. 그런데 도구를 준다고 끝이 아닙니다. **언제 어떤 도구를 부를지**, 그 **결과를 보고 다음에 무엇을 할지** 스스로 정해야 합니다.

**ReAct** 는 이 과정을 세 단계의 반복으로 풉니다.

| 단계 | 하는 일 | 예 |
|---|---|---|
| **생각(Reasoning)** | 지금 무엇을 할지 말로 정리 | "영업일을 세어 봐야겠다" |
| **행동(Acting)** | 도구 하나를 고르고 입력을 준다 | 영업일 도구에 `2026-05-08`, `5` |
| **관찰(Observation)** | 도구가 돌려준 결과를 받는다 | `2026-05-15` |

생각→행동→관찰을 **답이 나올 때까지 반복**합니다. 관찰이 새 정보를 주고, 그 정보로 다시 생각합니다. 이 반복이 에이전트를 "한 번 대답하고 끝"이 아니라 **여러 단계를 밟는 문제 해결자**로 만듭니다.

<img src="images/ReAct_루프.png" width="960">

> 화살표가 **한 바퀴 돌아 제자리로 돌아오는 것**을 보세요. 관찰이 끝이 아니라 다음 생각의 재료가 됩니다. 오른쪽 점선은 **탈출구**입니다 — 도구가 더 필요 없다고 판단하면 최종답을 내고 고리를 빠져나옵니다.

> **비유** — 요리사가 간을 보는 모습과 같습니다. "간을 봐야겠다"(생각) → 국물을 한 술 뜬다(행동) → "싱겁네"(관찰) → "소금을 넣자"(생각) … 결과를 **보고** 다음 행동을 정하는 것이 ReAct 입니다.

## 첫 도구는 모델이 모르는 것부터

도구가 왜 필요한지는 모델이 못 하는 일을 시켜 보면 가장 빨리 드러납니다. "영업일로 5일 뒤"는 **달력과 공휴일**을 알아야 답할 수 있습니다.

In [ ]:
# 도구 없이 모델에게 그냥 물어봅니다 - 답이 나오긴 합니다. 문제는 그다음입니다.
plain = model.invoke('2026-05-08 부터 5영업일 뒤는 며칠인가요? 한국 공휴일을 반영해서 알려 주세요.')
print(plain.text)

> 답이 그럴듯하게 나왔더라도 문제는 남습니다. **맞는지 확인할 길이 없습니다.** 그해 공휴일이 언제인지는 학습 이후에 정해질 수도 있고, 대체공휴일처럼 해마다 달라지는 것도 있습니다. 이런 값은 **바깥에서 받아 와야** 합니다.

먼저 응답이 어떻게 생겼는지 눈으로 봅니다. 데이터 수집 단원에서 배운 `requests` 그대로입니다 - 이 공휴일 서비스는 **키가 필요 없습니다.**

In [ ]:
# 도구로 감싸기 전에 응답을 눈으로 먼저 봅니다 - 데이터 수집 단원에서 배운 requests 그대로입니다.
import requests

res = requests.get('https://date.nager.at/api/v3/PublicHolidays/2026/KR', timeout=10)
res.raise_for_status()      # 4xx·5xx 면 여기서 멈춘다(조용한 실패를 만들지 않는다)
holidays = res.json()       # 이 서비스는 JSON 목록을 돌려준다

print('공휴일 수:', len(holidays))
print('앞의 셋 :', [(h['date'], h['localName']) for h in holidays[:3]])

이제 이 호출을 **도구로 감쌉니다.** 두 가지를 눈여겨보세요.

1. **docstring 이 모델에게 가는 설명서입니다.** *무엇을 하는지* 와 *언제 쓰는지* 를 적습니다.
2. **실패를 예외로 던지지 않고 문자열로 돌려줍니다.** 인터넷은 언제든 끊깁니다. 예외를 밖으로 던지면 에이전트가 그 자리에서 멈추지만, 문자열이면 모델이 그것을 읽고 사용자에게 안내합니다.

In [ ]:
# 그 요청을 도구로 만듭니다 - 안에서 공휴일 API 를 부르고, 주말·공휴일을 빼고 셉니다.
from datetime import date, timedelta

from langchain_core.tools import tool


WEEKDAY_NAMES = '월화수목금토일'    # date.weekday() 는 월요일이 0 이다


@tool
def business_day_after(start_date: str, days: int) -> str:
    """접수일(start_date, YYYY-MM-DD)부터 영업일(days)만큼 뒤가 며칠인지 알려준다.

    주말과 한국 공휴일은 세지 않는다. 처리 기한·완료 예정일을 물을 때 쓴다.
    """
    # 1) 받은 날짜부터 검사한다 - 모델이 '내일' 같은 말을 넘길 수도 있다.
    try:
        day = date.fromisoformat(start_date.strip())
    except ValueError:
        return f"'{start_date}' 는 날짜 형식이 아닙니다. 2026-05-08 처럼 적어 주세요."

    # 2) 공휴일 목록을 받아 온다. 시작 연도와 그다음 해까지 받는 이유는
    #    연말에 시작하면 영업일을 세다가 해를 넘기기 때문이다.
    holidays = {}
    for year in (day.year, day.year + 1):
        try:
            res = requests.get(f'https://date.nager.at/api/v3/PublicHolidays/{year}/KR', timeout=10)
            res.raise_for_status()               # 4xx·5xx 응답도 실패로 본다
        except requests.RequestException as error:
            # 실패도 '문자열' 로 돌려준다 - 예외를 밖으로 던지면 에이전트가 그 자리에서 멈춘다.
            return f'공휴일을 가져오지 못했습니다: {error}. 잠시 뒤 다시 시도해 주세요.'
        # {'2026-02-16': '설날', ...} 모양으로 모아 두면 날짜로 바로 찾을 수 있다.
        holidays.update({h['date']: h['localName'] for h in res.json()})

    # 3) 하루씩 넘기며 '세는 날' 이 days 개가 될 때까지 반복한다.
    #    주말·공휴일은 세지 않고, 왜 건너뛰었는지를 skipped 에 남긴다(답에 근거로 싣기 위해).
    left, skipped = days, []
    while left > 0:
        day += timedelta(days=1)
        weekday = WEEKDAY_NAMES[day.weekday()]
        holiday_name = holidays.get(day.isoformat())   # 공휴일이 아니면 None 이 나온다
        if day.weekday() >= 5:                         # 토(5)·일(6)
            reason = f'{weekday}요일'
            if holiday_name:                           # 공휴일이 주말과 겹치기도 한다
                reason = f'{weekday}요일·{holiday_name}'
            skipped.append(f'{day.isoformat()}({reason})')
            continue
        if holiday_name:                               # 평일인데 공휴일인 날
            skipped.append(f'{day.isoformat()}({holiday_name})')
            continue
        left -= 1                                      # 여기까지 왔으면 세는 날이다

    # 4) 답을 문장으로 만든다 - 건너뛴 날 목록은 줄을 바꿔 붙인다(한 줄에 몰면 읽기 어렵다).
    end = f'{day.isoformat()}({WEEKDAY_NAMES[day.weekday()]}요일)'
    if not skipped:
        return f'{start_date} 부터 {days}영업일 뒤는 {end} 입니다'
    return (f'{start_date} 부터 {days}영업일 뒤는 {end} 입니다\n'
            f"건너뛴 날 {len(skipped)}일: {', '.join(skipped)}")


print(business_day_after.invoke({'start_date': '2026-02-13', 'days': 3}))

> **`2026-02-13` 에서 3영업일 뒤가 `2026-02-23`** 입니다. 단순히 3일을 더하면 2월 16일이지만 그날은 설날 연휴이고, 주말까지 건너뛰면 23일이 됩니다. **바깥 세상을 봐야만 나오는 답**이라는 것이 이 도구의 존재 이유입니다.

> 도구가 **건너뛴 날을 날짜와 사유까지** 돌려주는 것도 눈여겨보세요(`2026-02-16(설날)`). 날짜 하나만 던져 주면 사용자도 모델도 그 값을 검산할 수 없습니다 - **왜 그 날짜인지**가 함께 와야 답변에 근거로 실을 수 있습니다. 주말과 공휴일이 겹친 날은 `2026-09-26(토요일·추석)` 처럼 둘 다 적습니다.

## 세 단계는 메시지 기록에 이렇게 남습니다
이제 도구를 에이전트에 붙이고, 지난 시간에 읽던 그 **메시지 기록**(`result['messages']` - 사람·모델·도구가 주고받은 메시지가 순서대로 쌓인 목록)을 다시 봅니다. 이번에는 **생각·행동·관찰이 각각 어느 메시지인지**를 짚어 봅니다.

In [ ]:
# 도구를 붙인 에이전트 - 이제 메시지 기록을 생각·행동·관찰이 어디에 남았는지 짚어 가며 읽습니다.
Q_DAYS = '2026-05-08 에 접수한 건이 보안검토 5영업일이면 언제 끝나?'

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

desk_agent = create_agent(model, [business_day_after],
                          system_prompt='너는 사내 AI 도입 상담 담당자다. 필요하면 도구를 써라.')
first = desk_agent.invoke({'messages': Q_DAYS})   # 루프가 끝날 때까지 돌고 결과를 돌려준다

# 메시지 하나를 한 덩어리로 끊어 찍습니다 - 도구 결과가 여러 줄이라 붙여 놓으면 읽기 어렵습니다.
for order, message in enumerate(first['messages'], start=1):
    print(f'[{order}] {type(message).__name__}')
    if isinstance(message, HumanMessage):
        print(f'     질문 : {message.text}')
    elif isinstance(message, AIMessage) and message.tool_calls:
        for call in message.tool_calls:
            print(f"     행동 : {call['name']}{call['args']}")
    elif isinstance(message, ToolMessage):
        # 도구가 돌려준 글은 여러 줄일 수 있어 줄마다 들여 찍습니다.
        for line in message.text.splitlines():
            print(f'     관찰 : {line}')
    else:
        print(f'     답변 : {message.text}')          # 도구를 부르지 않은 AIMessage = 사람에게 하는 말
    print()

> 출력에 찍힌 `질문 / 행동 / 관찰 / 답변` 네 라벨이 아래 표의 네 줄입니다.

| 기록에 남는 메시지 | 출력 라벨 | ReAct 단계 | 무엇이 담기나 |
|---|---|---|---|
| `HumanMessage` | 질문 | (시작) | 사용자가 물은 것 |
| `AIMessage` (`tool_calls` 있음) | 행동 | **생각 + 행동** | 무엇을 부를지 정한 결과. `text` 는 대개 비어 있고 **행동만** 담긴다 |
| `ToolMessage` | 관찰 | **관찰** | 도구가 실제로 돌려준 값 |
| `AIMessage` (`tool_calls` 없음) | 답변 | (탈출) | 도구가 더 필요 없다고 판단한 최종 답변 |

**생각이 문장으로 안 보이는 것**이 눈에 띄었을 겁니다. 요즘 모델은 생각을 문장으로 늘어놓지 않고 **바로 행동으로** 옮깁니다. 생각은 사라진 것이 아니라 **행동 선택 안에 접혀** 있습니다. 그래서 우리가 볼 수 있는 것은 "무엇을 골랐는가" 뿐이고, 그 선택을 좌우하는 것은 **도구의 설명**입니다 - 5절에서 실험으로 확인합니다.

## 실행 과정을 단계별로 들여다보기

`invoke` 는 **다 끝난 뒤 결과만** 줍니다. 중간에 무슨 일이 있었는지는 끝나고 나서야 볼 수 있지요. **`stream`** 은 **한 단계가 끝날 때마다** 그 시점의 상태를 하나씩 내보냅니다.

무엇을 내보낼지는 **`stream_mode`** 로 고릅니다. 같은 질문을 세 가지로 돌려 보고 화면이 어떻게 달라지는지 비교합니다.

| `stream_mode` | 한 번에 내보내는 것 | 무엇을 볼 때 쓰나 |
|---|---|---|
| `'messages'` | **모델이 만드는 글자 조각** | 답변이 써지는 모습을 화면에 흘릴 때 |
| `'updates'` (**기본값**) | **그 단계에서 새로 생긴 것만** | 방금 무슨 일이 있었는지 |
| `'values'` | **그때까지 쌓인 메시지 전체** | 지금까지의 대화 전부 |

먼저 **`'messages'`** 입니다. 챗봇 화면에서 글자가 하나씩 찍히는 그 모습이 이 모드입니다. 모델이 만들어 내는 **토큰 조각**이 도착하는 대로 오고, 그것이 어느 단계에서 나왔는지 알려 주는 `metadata` 가 짝으로 옵니다.

In [ ]:
# 1) stream_mode='messages' - 모델이 만드는 글자 조각이 도착하는 대로 옵니다.
Q_DAYS = '2026-05-08 에 접수한 건이 보안검토 5영업일이면 언제 끝나?'

for token, metadata in desk_agent.stream({'messages': Q_DAYS}, stream_mode='messages'):
    print(token.text, end='')

> 한 가지 눈여겨볼 점은 **중간 단계의 모델 호출도 함께 흘러나온다**는 것입니다. 도구를 부르기로 정하는 그 호출도 모델 호출이니까요(그때는 `token.text` 가 비어 있고 도구 이름이 조각으로 옵니다). 그래서 위 출력은 **최종 답변만** 이어진 것처럼 보이지만, 실제로는 빈 조각들이 먼저 지나간 뒤입니다. 화면에 특정 단계만 찍고 싶으면 `metadata` 를 보고 걸러 씁니다.

다음은 기본값인 **`'updates'`** 입니다. 여기서부터는 글자가 아니라 **메시지 단위**입니다. 쌓인 전체가 아니라 **방금 새로 생긴 것**만 오고, 그것을 누가 만들었는지(`model` 쪽인지 `tools` 쪽인지)가 열쇠로 함께 옵니다. `stream_mode` 를 아예 빼고 `desk_agent.stream({'messages': Q_DAYS})` 라고만 써도 **결과가 같습니다**.

In [ ]:
# 2) stream_mode='updates' - 기본값입니다. 그 단계에서 새로 생긴 메시지만 옵니다.
for order, chunk in enumerate(desk_agent.stream({'messages': Q_DAYS}, stream_mode='updates'), start=1):
    for made_by, payload in chunk.items():          # made_by: 'model' 또는 'tools'
        kinds = [type(message).__name__ for message in payload['messages']]
        print(order, '번째 :', made_by, '에서 새로 생김', kinds)

마지막은 **`'values'`** 입니다. 매번 **그때까지 쌓인 메시지 전부**가 오므로, 개수가 1개 → 2개 → 3개 → 4개로 늘어납니다.

In [ ]:
# 3) stream_mode='values' - 그때까지 쌓인 메시지 전부가 옵니다. 개수가 한 개씩 늘어납니다.
for order, snapshot in enumerate(desk_agent.stream({'messages': Q_DAYS}, stream_mode='values'), start=1):
    print(order, '번째 : 쌓인 메시지', len(snapshot['messages']), '개')

> 세 실행을 나란히 놓고 보세요. `'messages'` 는 **글자 조각**이 수십 번, `'updates'` 는 **세 번**(모델 → 도구 → 모델), `'values'` 는 **네 번**(질문까지 포함해 쌓인 전체) 나옵니다.

여기서 하나 짚고 갑니다. **`'updates'`·`'values'` 는 메시지 단위**라 한 단계가 끝나야 그 단계에서 만들어진 메시지가 **통째로** 나옵니다. 최종 답변도 예외가 아니어서, 모델이 문장을 다 만든 다음 **한 번에** 옵니다. 답변이 써지는 모습을 그대로 보여 주고 싶다면 그때 쓰는 것이 `'messages'` 입니다.

우리는 앞으로 **`'values'`** 를 씁니다. 목록이 한 줄씩 길어지는 모습이 곧 "루프가 돌면서 기록이 쌓인다" 는 이야기라서, 지금 배우는 내용과 그림이 맞기 때문입니다. 이제 **맨 끝 메시지에 라벨을 붙여** 다시 읽어 봅니다.

In [ ]:
# agent.stream - 값이 쌓이는 모습이 곧 루프의 진행입니다(스냅샷 하나 = 그때까지 쌓인 메시지 전부).
Q_DAYS = '2026-05-08 에 접수한 건이 보안검토 5영업일이면 언제 끝나?'

for order, snapshot in enumerate(desk_agent.stream({'messages': Q_DAYS}, stream_mode='values'), start=1):
    latest = snapshot['messages'][-1]          # 그때까지 쌓인 것 중 맨 끝 = 방금 일어난 일
    print(f"[{order}번째] 쌓인 메시지 {len(snapshot['messages'])}개 · 맨 끝은 {type(latest).__name__}")
    if isinstance(latest, HumanMessage):
        print(f'     질문 : {latest.text}')
    elif isinstance(latest, AIMessage) and latest.tool_calls:
        print(f"     행동 : {[call['name'] for call in latest.tool_calls]}")
    elif isinstance(latest, ToolMessage):
        print(f'     관찰 : {latest.text.splitlines()[0]}')   # 여러 줄이면 첫 줄만
    else:
        print(f'     답변 : {latest.text[:60]}')
    print()

> 스냅샷이 하나씩 늘어나는 것을 보세요. 첫 스냅샷에는 **아직 질문 하나뿐**입니다. 그 뒤로 `AIMessage`(행동) → `ToolMessage`(관찰) → `AIMessage`(답변), **세 단계가 그대로** 지나갑니다.

## 메시지 기록에서 숫자를 뽑아 두기
메시지 기록을 매번 눈으로 읽을 수는 없습니다. **무엇을 몇 번 불렀고 모델이 몇 번 말했는지**를 뽑는 작은 헬퍼를 만들어 두면, 뒤에서 도구를 고칠 때마다 **바뀐 것을 수치로** 볼 수 있습니다.

In [ ]:
# 기록을 읽는 작은 헬퍼 둘 - 무엇을 몇 번 불렀고 모델이 몇 번 말했나.
from langchain_core.messages import AIMessage


def tool_names(result):
    """메시지 기록에서 실제로 불린 도구 이름 목록을 뽑는다."""
    names = []
    for message in result['messages']:
        if isinstance(message, AIMessage):   # 도구 호출은 AIMessage 에만 담긴다
            for call in message.tool_calls:
                names.append(call['name'])
    return names


def step_count(result):
    """모델이 몇 번 말했는지 센다(AIMessage 수). 도구를 거칠수록 늘어난다."""
    return sum(1 for message in result['messages'] if isinstance(message, AIMessage))


print('불린 도구       :', tool_names(first))
print('모델이 말한 횟수 :', step_count(first))
print('쌓인 메시지 수   :', len(first['messages']))

> 이 두 헬퍼 - **`tool_names`·`step_count`** - 는 오늘 끝까지 씁니다. 4절에서는 라우팅이 맞는지, 5절에서는 설명을 바꿨을 때 선택이 달라지는지를 이걸로 확인합니다.

### 🖐️ 함께 따라하기: 다음 공휴일을 알려 주는 도구

같은 공휴일 서비스에는 **다른 창구(엔드포인트)** 도 있습니다. `https://date.nager.at/api/v3/NextPublicHolidays/KR` 를 부르면 **앞으로 남은 공휴일**이 옵니다. 이번에는 그것을 도구로 감싸 보세요.

1. `requests.get(...)` 으로 그 주소를 한 번 불러 **응답이 어떻게 생겼는지** 먼저 출력해 보세요.
2. `@tool` 로 **`next_holidays(count: int) -> str`** 를 만드세요 — 앞에서 `count` 개만 골라 `날짜(이름)` 형태의 한 줄 문자열로 돌려줍니다. docstring 에 *언제 쓰는 도구인지* 를 적으세요.
3. 위에서 만든 도구처럼 **실패를 문자열로** 돌려주게 `try` / `except requests.RequestException` 을 넣으세요.
4. `create_agent(model, [next_holidays])` 로 에이전트를 만들어 `'앞으로 다가올 공휴일 3개만 알려줘'` 를 `invoke` 하고, `tool_names` 와 최종 답을 출력하세요.

**확인 기준**: 도구가 한 번 불리고, 답에 공휴일 이름이 들어 있습니다. **같은 서비스라도 창구가 다르면 도구를 따로 만든다** — 도구는 "모델이 고를 수 있는 행동 하나" 이기 때문입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) requests.get 으로 NextPublicHolidays/KR 을 한 번 불러 응답을 출력한다
# 2) @tool 로 next_holidays(count: int) -> str 를 만든다 (앞에서 count 개만)
#    docstring 에 '언제 쓰는 도구인지' 를 적는다
# 3) 실패는 예외로 던지지 말고 문자열로 돌려준다
# 4) create_agent 로 붙여 '앞으로 다가올 공휴일 3개만 알려줘' 를 invoke 하고
#    tool_names 와 최종 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** ReAct 의 세 단계는 무엇이고, 메시지 기록에서 **관찰**은 어느 메시지인가요?

<details><summary>정답 보기</summary>

**생각(Reasoning) → 행동(Acting) → 관찰(Observation)** 입니다. 관찰은 **`ToolMessage`** — 도구가 실제로 돌려준 값이 담깁니다. 그 앞의 `AIMessage` 에는 **부르기로 한 결정**(`tool_calls`)이 담깁니다.

</details>

**2.** 외부 API 도구에서 실패를 예외로 던지지 않고 문자열로 돌려주는 이유는?

<details><summary>정답 보기</summary>

예외를 밖으로 던지면 **에이전트가 그 자리에서 멈춥니다.** 문자열로 돌려주면 모델이 그것을 관찰로 읽고 사용자에게 안내하거나 다시 시도합니다. 인터넷은 언제든 끊기므로 외부 API 도구에서는 특히 중요합니다.

</details>

**3.** 모델이 도구를 부를 때 `AIMessage` 의 `text` 가 비어 있는 이유는?

<details><summary>정답 보기</summary>

생각이 **행동 선택 안에 접혀** 있기 때문입니다. 요즘 모델은 생각을 문장으로 늘어놓지 않고 바로 `tool_calls` 로 행동을 표현합니다. 그래서 우리가 볼 수 있는 것은 "무엇을 골랐는가" 이고, 그 선택을 좌우하는 것은 도구의 설명입니다.

</details>

---
# 2. 문서를 검색하는 도구 만들기

## 긴 문서는 조각으로 찾습니다
오늘 검색할 자료는 **74쪽짜리 안내서 두 건**입니다. 한 쪽을 통째로 색인하면 어떻게 될까요? 한 쪽에는 여러 이야기가 섞여 있어서, 질문과 관계있는 두 문장 때문에 **관계없는 열 문장까지** 함께 딸려 옵니다. 그래서 지난 단원에서 배운 대로 **조각(청크)** 으로 자릅니다.

이 벡터DB 는 아래 코드로 **미리 만들어 두었습니다.** 자르는 규칙은 지난 단원에서 배운 그대로이고, 새로 붙는 것은 꼬리표의 **조각 번호**입니다. (읽어만 보세요 — 이미 만들어져 있으니 실행할 필요가 없습니다.)

```python
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)

chunks = []
for row in guide_df.itertuples():                 # 안내서 한 쪽이 한 행
    parts = splitter.split_text(row.본문)
    for i, part in enumerate(parts):
        chunks.append(Document(page_content=part, metadata={
            'doc_id': row.id, 'chunk_no': i, 'chunk_total': len(parts),
            '쪽': int(row.쪽), '소제목': str(row.소제목)}))

Chroma.from_documents(chunks, embeddings, collection_name='policy_day20',
                      persist_directory='data/chroma_day20',
                      ids=[f"{d.metadata['doc_id']}-{d.metadata['chunk_no']}" for d in chunks])
```

> `persist_directory` 를 주면 색인이 **폴더에 남습니다.** 그래서 우리는 여는 것으로 시작할 수 있었지요. 자르는 규칙(`chunk_size`·`chunk_overlap`)을 바꾸고 싶다면 이 코드를 다시 돌려 벡터DB 를 새로 만들면 됩니다.

**꼬리표에 새로 붙은 것**을 보세요 — 오늘 쓰는 배관의 핵심입니다.

| 꼬리표 | 무엇 | 왜 필요한가 |
|---|---|---|
| `doc_id` | 어느 쪽에서 나왔나 | 같은 쪽의 다른 조각을 찾을 열쇠 |
| `chunk_no` | 그 쪽의 **몇 번째** 조각인가 | **앞뒤 조각**을 지목하는 번호 |
| `chunk_total` | 그 쪽이 몇 조각으로 나뉘었나 | 처음·끝을 넘어가지 않게 |
| `쪽` · `소제목` | 출처 | 답에 **근거**를 붙일 때 |

> 안내서 42쪽이 **194개 조각**이 되었습니다(한 쪽당 3~7개). 조각 하나는 평균 300자 남짓입니다.

In [ ]:
# 조각 하나를 그대로 들여다봅니다 - 가운데 조각은 어디서 시작해 어디서 끊길까요?
from langchain_core.documents import Document

# 6쪽(ai6)의 1번 조각을 꺼냅니다 - 저장소에서 조건으로 꺼내는 방법은 잠시 뒤에 그대로 다시 씁니다.
got = policy_store.get(where={'$and': [{'doc_id': {'$eq': 'ai6'}}, {'chunk_no': {'$eq': 1}}]})
middle = Document(page_content=got['documents'][0], metadata=got['metadatas'][0])

print('꼬리표:', middle.metadata)
print('길이  :', len(middle.page_content))
print('시작  :', repr(middle.page_content[:60]))
print('끝    :', repr(middle.page_content[-60:]))

> **문장 한가운데에서 시작하고, 한가운데에서 끝납니다.** 자를 때 겹침(`chunk_overlap`)을 주었는데도 그렇습니다. 겹침은 경계에 걸린 **한 문장**을 살릴 뿐, 그 문장이 딛고 선 **앞의 설명**까지 가져오지는 못하기 때문입니다.

## 그래서 앞뒤를 함께 가져옵니다
검색은 **조각 단위로** 하는 것이 맞습니다 - 조각이 작아야 질문과 정확히 맞는 대목이 잡히니까요. 다만 **모델에게 넘길 때는** 그 조각 혼자가 아니라 **앞뒤 조각까지 붙여** 넘깁니다.

| | 찾을 때 | 넘길 때 |
|---|---|---|
| 단위 | **작은 조각** - 정확히 맞는 대목이 잡힌다 | **조각 + 앞뒤** - 문맥이 이어진다 |
| 이유 | 조각이 크면 관계없는 글이 섞여 순위가 흐려진다 | 조각만 주면 문장이 잘려 뜻이 반쪽이 된다 |

<img src="images/앞뒤_문맥_확장.png" width="900">

*검색이 집은 조각은 하나지만, 모델에게는 그 앞뒤까지 이어 붙여 넘깁니다.*

> 꼬리표에 `chunk_no` 를 남겨 둔 것이 여기서 값을 합니다. **"같은 `doc_id` 이면서 `chunk_no` 가 앞뒤인 것"** 을 컬렉션에서 꺼내 오면 됩니다.

In [ ]:
# 앞뒤 조각을 같은 문서에서 꺼내 이어 붙이는 함수.
def with_neighbors(store, hit, window=1):
    """검색된 조각의 앞뒤 window개를 같은 문서에서 꺼내 순서대로 이어 붙인다."""
    tag = hit.metadata
    # 필요한 조각 번호 목록 - 문서의 처음·끝을 넘어가지 않게 자른다.
    want = [n for n in range(tag['chunk_no'] - window, tag['chunk_no'] + window + 1)
            if 0 <= n < tag['chunk_total']]
    got = store.get(where={'$and': [{'doc_id': {'$eq': tag['doc_id']}},
                                    {'chunk_no': {'$in': want}}]})
    # get 은 순서를 보장하지 않는다 - 조각 번호로 다시 줄을 세워야 글이 이어진다.
    pairs = sorted(zip(got['metadatas'], got['documents']), key=lambda p: p[0]['chunk_no'])
    return '\n'.join(text for _tag, text in pairs)


# 같은 조각을 '혼자' 볼 때와 '앞뒤까지' 볼 때의 길이를 견줍니다.
print('조각만  :', len(middle.page_content), '자')
print('앞뒤까지:', len(with_neighbors(policy_store, middle)), '자')

In [ ]:
# 실제 검색 결과에 적용해 봅니다 - 1등 조각을 혼자 볼 때와 앞뒤까지 볼 때.
Q_POLICY = '이용자가 챗봇에 입력한 대화를 학습에 쓰려면 무엇을 알려야 하나요?'

hits = policy_retriever.invoke(Q_POLICY)      # 준비 셀에서 k=3 으로 만들어 둔 검색기

top = hits[0]                                 # 1등 = 질문과 가장 가까운 조각
# 꼬리표만 봐도 어느 쪽 몇 번째 조각인지 알 수 있다 - 출처 표기의 재료다.
print('1등 조각:', f"{top.metadata['doc_id']}-{top.metadata['chunk_no']}",
      f"({top.metadata['쪽']}쪽 · {top.metadata['소제목']})")
print()
print('--- 조각만 ---')
print(top.page_content)
print()
print('--- 앞뒤까지 (window=1) ---')
print(with_neighbors(policy_store, top))

> 두 출력을 나란히 읽어 보세요. 조각만 볼 때는 **"- 2) LLM 서비스는 …"** 처럼 앞뒤가 잘린 채 시작하지만, 앞뒤를 붙이면 **무엇에 대한 이야기인지**가 드러납니다.

> **`window` 는 앞뒤로 몇 조각을 더 붙일지 정하는 파라미터입니다.** 크게 잡으면 문맥이 넉넉해지는 대신 프롬프트가 길어지고 (비용도 늘고) 관계없는 글이 섞입니다. 여기서는 앞뒤 한 개(`window=1`)로 둡니다.

이제 검색 결과 전체를 **출처가 붙은 한 덩어리 글**로 만듭니다. 이 함수를 체인도 쓰고 도구도 쓰게 해서, 둘이 조용히 어긋나는 일을 막습니다.

In [ ]:
# 검색 결과 전체를 '출처가 붙은 한 덩어리 글' 로 만듭니다 - 체인과 도구가 함께 씁니다.
def policy_context(store, hits, window=1):
    """검색된 조각마다 출처(쪽·소제목)를 붙이고 앞뒤 문맥까지 이어 한 덩어리로 만든다."""
    blocks = []
    for hit in hits:
        tag = hit.metadata
        # 조각마다 [출처] 한 줄을 머리에 얹고, 본문은 앞뒤까지 넓혀 붙인다.
        #   출처를 글 안에 넣어 두면 모델이 그 번호를 답에 그대로 옮겨 적을 수 있다.
        blocks.append(f"[{tag['쪽']}쪽 · {tag['소제목']}]\n"
                      + with_neighbors(store, hit, window))
    return '\n\n'.join(blocks)      # 조각 사이는 빈 줄로 띄운다


print(policy_context(policy_store, hits)[:300], '...')

## 미뤄 둔 문제: 체인은 인사말에도 검색이 돈다

지난 단원에서 만든 RAG **체인**에는 성질이 하나 있었습니다. **무엇을 묻든 반드시 검색합니다.** 파이프로 고정된 경로이기 때문이지요. 그 장면을 다시 봅니다.

<img src="images/체인_vs_에이전트_검색.png" width="820">

*체인은 경로가 고정돼 언제나 검색을 지나가고, 에이전트는 검색이 필요한지부터 모델이 판단한다.*

In [ ]:
# 앞 단원에서 만든 그 체인입니다 - 검색 -> 프롬프트 -> 모델을 파이프로 고정한 것.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 자료가 들어갈 자리({context})와 질문 자리({question})를 둔 프롬프트.
policy_prompt = ChatPromptTemplate.from_template(
    '너는 사내 AI 도입 상담 담당자다. 주어진 자료에 있는 내용만으로 한국어로 답하라.\n\n'
    '자료:\n{context}\n\n질문: {question}')

policy_chain = (
    # 같은 질문이 두 갈래로 동시에 들어간다.
    #   context 갈래: 질문 -> 검색 -> 출처와 앞뒤 문맥이 붙은 한 덩어리 글
    #   question 갈래: 질문을 그대로 통과시킨다
    {'context': RunnableLambda(lambda q: policy_context(policy_store, policy_retriever.invoke(q))),
     'question': RunnablePassthrough()}
    | policy_prompt | model | StrOutputParser())      # 두 자리를 채워 모델에 보내고 글만 뽑는다

print('체인 준비 완료')

In [ ]:
# 인사말을 체인에 넣어 봅니다 - 경로가 고정이라 검색이 '반드시' 한 번 돕니다.
GREETING = '고마워요, 오늘도 수고 많으세요!'

greet_hits = policy_retriever.invoke(GREETING)
print('인사말로 검색된 조각:', [f"{d.metadata['쪽']}쪽 {d.metadata['소제목'][:16]}" for d in greet_hits])
print()
print('체인의 답:', policy_chain.invoke(GREETING))

> 인사말과 아무 상관 없는 대목이 자료로 붙었습니다. **벡터 검색은 '관련 없음' 이라고 답하지 않습니다** - 언제나 *상대적으로* 가장 가까운 `k` 개를 돌려줍니다. 체인은 그 결과를 무조건 프롬프트에 넣습니다.

## 해결책은 검색을 도구로 감싸는 것
1절에서 본 것을 떠올려 보세요. 에이전트는 **도구를 쓸지 말지 모델이 스스로 정합니다.** 그러니 검색을 `@tool` 로 감싸 붙이면, **검색이 필요한 질문에만** 검색이 돕니다.

In [ ]:
# 해결 - 검색을 도구로 감쌉니다. docstring 이 곧 모델이 읽는 사용 설명서입니다.
@tool
def search_policy(query: str) -> str:
    """생성형 AI 개인정보 처리 안내서에서 질문과 관련된 규정 대목을 찾아 돌려준다.

    학습데이터·고지·동의·국외이전·위탁처럼 AI 서비스를 만들거나 도입할 때 지켜야 할
    개인정보 규정을 물을 때 쓴다. 결과에는 몇 쪽에서 왔는지가 함께 붙는다.
    """
    # 도구 안은 방금 만든 함수 두 개를 이어 부르는 것이 전부다 - 검색하고, 출처 붙여 합친다.
    return policy_context(policy_store, policy_retriever.invoke(query))


policy_agent = create_agent(model, [search_policy])   # 도구는 하나뿐이지만 쓸지 말지는 모델이 정한다
print('도구 설명 첫 줄:', search_policy.description.splitlines()[0])

In [ ]:
# 같은 두 입력을 에이전트에 넣어 체인과 견줍니다 - 달라지는 것은 '검색을 도는가' 입니다.
Q_POLICY = '이용자가 챗봇에 입력한 대화를 학습에 쓰려면 무엇을 알려야 하나요?'
GREETING = '고마워요, 오늘도 수고 많으세요!'

for q in [Q_POLICY, GREETING]:
    res = policy_agent.invoke({'messages': q})
    print(f'질문: {q}')
    # 호출 수가 이 실험의 눈금이다 - 규정 질문은 1회, 인사말은 0회가 나와야 한다.
    print(f'  도구 호출 수: {len(tool_names(res))}')
    print(f'  답          : {res["messages"][-1].text[:80]}')
    print()

> **같은 인사말인데 결과가 다릅니다.**

| | 지난 단원의 RAG 체인 | 오늘의 에이전트 |
|---|---|---|
| 인사말을 넣으면 | 검색이 **반드시** 돈다 | 도구 호출 **0회** |
| 붙는 근거 | 관계없는 대목 세 개 | 없음 |
| 규정 질문 | 검색 후 답 | 검색 도구를 **골라** 부른 뒤 답 |
| 경로를 정하는 주체 | **우리**(파이프로 고정) | **모델**(질문을 보고 판단) |

**그럼 체인은 쓸모없을까요? 아닙니다.** 들어오는 질문이 전부 문서 검색으로 답할 것이라면 체인이 낫습니다 - 경로가 고정되어 **동작이 예측 가능하고**, 모델을 한 번만 부르니 **빠르고 저렴**합니다. 에이전트는 판단을 위해 모델을 한 번 더 부르는 대신 **다양한 질문을 받아 낼 수 있습니다.** 무엇이 들어올지 모르는 창구라면 에이전트, 정해진 흐름이라면 체인 - 이렇게 고르면 됩니다.

## 검색 도구를 붙였으면 근거와 출처는 기본입니다

검색 도구를 붙였어도 에이전트가 **찾은 내용을 무시하고 지어내면** 소용이 없습니다. 개인정보 규정을 잘못 안내하면 실제 사고로 이어집니다. 그래서 문서 검색을 붙일 때는 시스템 프롬프트로 두 가지를 **함께** 못박습니다 — **찾은 내용에만 근거할 것**, 그리고 **출처(쪽 번호)를 밝힐 것**.

> 앞에서 검색 결과에 `[NN쪽 · 소제목]` 을 붙여 둔 것이 여기서 값을 합니다. 모델이 읽는 자료 안에 쪽 번호가 들어 있으니, **그 번호를 그대로 답에 옮겨 적게** 할 수 있습니다.

In [ ]:
# 근거에만 기대어 답하게 - 시스템 프롬프트로 못박고 출처(쪽 번호)를 붙이게 합니다.
Q_POLICY = '이용자가 챗봇에 입력한 대화를 학습에 쓰려면 무엇을 알려야 하나요?'
Q_NO_GROUND = '우리 회사 연차 휴가는 며칠 전까지 신청해야 하나요?'

# 프롬프트에 세 가지를 못박는다 - (1) 도구로 찾을 것 (2) 찾은 내용에만 근거할 것 (3) 쪽 번호를 붙일 것.
cited_agent = create_agent(
    model, [search_policy],
    system_prompt=('너는 사내 AI 도입 상담 담당자다. 반드시 search_policy 로 찾은 내용에만 근거해 답하고, '
                   '찾은 내용이 없으면 모른다고 답하라. 답 끝에 근거로 삼은 쪽 번호를 '
                   '"(근거: 생성형AI 안내서 NN쪽)" 형식으로 붙여라.'))

# 자료에 있는 질문과 없는 질문을 나란히 - 없는 쪽은 '모른다' 고 답해야 맞습니다.
for q in [Q_POLICY, Q_NO_GROUND]:
    r = cited_agent.invoke({'messages': q})
    print('Q:', q)
    print('A:', r['messages'][-1].text[:220])
    print()

> 있는 질문에는 답 끝에 **쪽 번호**가 붙고, 이 안내서에 없는 연차 규정에는 **모른다**고 답합니다. 검색은 없는 질문에도 무언가를 돌려준다는 것(가장 가까운 `k` 개)을 기억하세요 - 그러니 마지막 방어선은 **"찾은 내용에만 근거하라"** 는 지시입니다.

> 다만 이것은 **부탁**이라는 점을 기억하세요. 프롬프트로 근거를 요구하는 것은 코드로 막는 것과 다릅니다. 정말 확실히 하려면 **답을 내보내기 전에 근거가 실제로 있었는지 검사**해야 합니다 - 그 이야기는 신뢰성·가드레일 단원에서 이어집니다.

**앞으로 이 노트북의 문서 검색 에이전트는 이 프롬프트를 기본으로 답니다.**

### 🖐️ 함께 따라하기: 다른 안내서로 검색 도구 만들기

준비 셀이 **「개인정보 처리방침 작성지침」** 도 색인해 두었습니다(`std_store`·`std_retriever`). 이 문서로 같은 절차를 처음부터 해 보세요.

1. `'개인정보 처리방침에는 무엇을 적어야 하나요?'` 로 `std_retriever` 를 검색해, 1등 조각의 **쪽·소제목·조각 번호**를 출력하세요.
2. 같은 조각을 `with_neighbors(std_store, 조각)` 로 넓혀 **길이가 얼마나 늘었는지** 견주세요 (`with_neighbors` 는 저장소를 인자로 받으므로 **함수는 고치지 않습니다**).
3. `@tool` 로 **`search_standard(query: str) -> str`** 를 만드세요 — 안에서 `policy_context(std_store, std_retriever.invoke(query))` 를 돌려주고, docstring 에 *언제 쓰는 도구인지*(처리방침 작성·공개 항목 같은 주제)를 적습니다.
4. `create_agent(model, [search_standard], system_prompt=...)` 로 에이전트를 만드세요 — 프롬프트에 **찾은 내용에만 근거할 것**, 없으면 **모른다고 할 것**, 답 끝에 **`(근거: 처리방침 작성지침 NN쪽)`** 을 붙일 것을 적습니다(방금 데모와 같은 방식입니다).
5. 같은 질문과 **인사말**(`GREETING`)을 각각 `invoke` 해 두 경우의 `tool_names` 를 출력해 견주고, 규정 질문의 답에 **쪽 번호**가 붙었는지 확인하세요.

**확인 기준**: 규정 질문에는 도구가 불리고 답에 쪽 번호가 붙으며, 인사말에는 호출 수가 0 입니다. **자료가 바뀌어도 절차는 그대로**라는 것이 이 연습의 요점입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) std_retriever 로 '개인정보 처리방침에는 무엇을 적어야 하나요?' 를 검색해
#    1등 조각의 쪽·소제목·조각 번호를 출력한다
# 2) with_neighbors(std_store, 조각) 로 넓혀 길이를 견준다
# 3) @tool 로 search_standard(query) 를 만든다 (policy_context + std_store)
# 4) create_agent 로 붙여 규정 질문과 인사말의 tool_names 를 견준다

### ✅ 바로 확인 퀴즈

**1.** 찾을 때는 작은 조각으로 하면서, 모델에게 넘길 때는 앞뒤를 붙이는 이유는?

<details><summary>정답 보기</summary>

**목적이 다르기 때문**입니다. 찾을 때는 조각이 작아야 질문과 맞는 대목이 정확히 잡히고(크면 관계없는 글이 섞여 순위가 흐려집니다), 읽을 때는 앞뒤가 있어야 문장이 이어져 뜻이 통합니다. 그래서 **검색 단위와 전달 단위를 다르게** 둡니다.

</details>

**2.** 앞뒤 조각을 꺼내오려면 꼬리표에 무엇이 있어야 하나요?

<details><summary>정답 보기</summary>

**`doc_id`·`chunk_no`·`chunk_total`** 입니다. 같은 문서(`doc_id`)에서 번호가 앞뒤인 조각을 고르고, `chunk_total` 로 처음·끝을 넘어가지 않게 막습니다. 꺼낸 결과는 순서가 보장되지 않으므로 **`chunk_no` 로 다시 정렬**해야 글이 이어집니다.

</details>

**3.** 답에 쪽 번호를 붙일 수 있었던 이유는 무엇인가요?

<details><summary>정답 보기</summary>

검색 결과를 한 덩어리 글로 만들 때 **`[NN쪽 · 소제목]` 을 함께 넣었기** 때문입니다. 조각의 꼬리표(`metadata`)에 출처를 담아 둔 것이 여기서 쓰입니다 — 모델은 자기가 받은 자료 안에 있는 번호를 옮겨 적을 뿐입니다.

</details>

**4.** 인사말을 RAG 체인에 넣으면 왜 엉뚱한 자료가 붙나요?

<details><summary>정답 보기</summary>

체인은 경로가 고정되어 **무조건 검색을 지나가고**, 벡터 검색은 '관련 없음' 을 돌려주지 않고 언제나 **상대적으로 가장 가까운 `k` 개**를 돌려주기 때문입니다. 검색을 도구로 감싸면 **검색 여부를 모델이 판단**합니다.

</details>

---
# 3. 표를 조회하는 도구 만들기

## 문서로는 답할 수 없는 질문
"국외로 데이터가 나가는 서비스가 **몇 개**야?" 는 안내서를 아무리 뒤져도 안 나옵니다. **세어 봐야** 알 수 있고, 그 숫자는 **우리 회사 표** 안에 있습니다. 그것을 세는 언어가 SQL 단원에서 배운 **SQL** 이지요.

지난 단원에서 만든 그 구조를 그대로 씁니다.

1. **표 구조를 프롬프트로 알려 준다** (어떤 표에 어떤 열이 있는지)
2. **모델이 SQL 을 만든다**
3. **우리가 그 SQL 을 실행한다**

> **모델은 데이터베이스를 볼 수 없습니다.** 표가 몇 개인지, 열 이름이 무엇인지 — 전부 **우리가 프롬프트에 적어 준 것만** 압니다. **스키마 설명이 곧 모델의 눈입니다.**

In [ ]:
# [제공 코드] 데이터베이스 준비 - SQL 단원에서 배운 sqlite 를 그대로 씁니다(접속 정보가 필요 없습니다).
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('output') / 'ai_desk.db'
DB_PATH.parent.mkdir(exist_ok=True)
DB_PATH.unlink(missing_ok=True)          # 여러 번 실행해도 늘 같은 초기 상태에서 시작합니다

_conn = sqlite3.connect(DB_PATH, isolation_level=None)   # isolation_level=None : 실행 즉시 저장
_conn.execute('pragma foreign_keys = on')                # 외래키 검사를 켭니다(기본값은 꺼짐)
_conn.executescript(Path('data/setup_day20_agent.sql').read_text(encoding='utf-8'))
_conn.execute('pragma foreign_keys = on')                # executescript 뒤에 한 번 더 켭니다

# 에이전트에게 줄 연결은 따로 만들고 '읽기 전용'으로 엽니다 - 모델이 무슨 SQL 을 만들든 쓰기가 막힙니다.
#  check_same_thread=False : 에이전트는 도구를 별도 스레드에서 실행하므로 이 옵션이 없으면 도구가 전부 실패합니다.
_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True,
                           isolation_level=None, check_same_thread=False)


def run_query(sql):
    """SELECT 결과를 DataFrame 으로 돌려준다(사람이 눈으로 확인할 때 쓴다)."""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print('데이터베이스 준비 완료 -', DB_PATH)

질문을 만들기 전에 **어떤 데이터가 있는지 먼저 봅니다.**

In [ ]:
# 사내에 도입했거나 검토 중인 AI 서비스 목록.
display(run_query('select * from ai_service'))

In [ ]:
# 부서별 사용 신청 내역 -- service_id 로 위의 서비스 표와 이어집니다(외래키).
display(run_query('select * from ai_request'))

## ⚠️ 남이 만든 SQL 을 실행한다는 것

**모델이 만든 SQL 을 그대로 실행한다**는 말을 다시 읽어 보세요. 모델이 악의를 갖는다는 뜻이 아닙니다 — 질문을 잘못 알아듣고 `delete` 를 만들 수도 있고, 사용자가 질문에 교묘한 지시를 섞어 넣을 수도 있습니다. 그래서 가드를 **두 겹**으로 겁니다.

| 겹 | 무엇을 하나 | 무엇을 막나 |
|---|---|---|
| 1) 문자열 검사 | `select` 로 시작하는 **한 문장**만 통과 | 대놓고 쓰는 `delete`·여러 문장 |
| 2) 읽기 전용 연결 | 연결 자체가 쓰기를 거부 | **1겹을 빠져나간 모든 쓰기** |

In [ ]:
# [제공 코드] 데이터베이스 조회 도구 - 에이전트가 이 도구로 SQL 을 실행합니다.
#  가드가 두 겹입니다: (1) 여기서 문장을 검사하고 (2) 연결 자체가 읽기 전용입니다.
from langchain_core.tools import tool


@tool
def run_select(sql: str) -> str:
    """읽기 전용 SQL(SELECT) 한 문장을 실행하고 결과를 문자열로 돌려준다. SELECT 한 문장이 아니면 거부한다."""
    stmt = sql.strip().rstrip(';')          # 끝의 세미콜론 하나는 흔한 표기라 허용한다
    # 세미콜론이 남아 있으면 문장이 둘 이상이라는 뜻 - 'select 1; delete ...' 를 막는다.
    if not stmt.lower().startswith('select') or ';' in stmt:
        return '거부: 이 도구는 SELECT 조회 한 문장만 실행할 수 있습니다.'
    try:
        return str(_ro_conn.execute(stmt).fetchall())   # 검사한 문장을 그대로 실행한다
    except Exception as e:
        return f'에러: {e}'                             # 에러도 문자열로 - 모델이 읽고 고쳐 다시 시도한다


print('SQL 도구 준비:', run_select.name)

In [ ]:
# 첫 번째 겹(문자열 검사)이 무엇을 막고 무엇을 통과시키는지 직접 확인합니다.
attempts = [
    'select count(*) from ai_service',                        # 평범한 조회
    'select 1; delete from ai_request',                       # 조회인 척하며 뒤에 삭제를 붙임
    'delete from ai_request',                                 # 대놓고 삭제
    'select 1 from ai_service where 1=0 union select 1',      # 조회이지만 복잡한 문장
]
for sql in attempts:
    print(sql)
    print('   ->', run_select.invoke({'sql': sql}))

> 앞의 셋은 예상대로입니다. 문제는 **네 번째**입니다. `select 1 from ai_service where 1=0 union select 1` 은 **문자열 검사를 통과합니다** - `select` 로 시작하고 세미콜론도 없으니까요.

**금지 목록을 늘리는 방식은 언제나 빠져나갈 구멍이 남습니다.** 그래서 두 번째 겹이 필요합니다. 이번에는 도구를 거치지 않고 **읽기 전용 연결에 직접** 삭제를 시도해 봅니다.

In [ ]:
# 문자열 검사를 건너뛰고 읽기 전용 연결에 직접 삭제를 시도합니다.
#   try 로 감싼 이유: 여기서 에러가 나는 것이 '정상' 이고, 그 에러 문구를 보여 주려는 것입니다.
try:
    _ro_conn.execute('delete from ai_request')
    print('삭제되었습니다 -- 이러면 안 됩니다')
except Exception as e:
    print('막혔습니다:', e)

print('남은 신청 건수:', run_query('select count(*) as cnt from ai_request').loc[0, 'cnt'])

> **`attempt to write a readonly database`** - 연결 자체가 쓰기를 거부했습니다. 문자열 검사는 **막을 것을 하나씩 골라내는** 방식이라 새로운 수법이 나오면 뚫리지만, 읽기 전용 연결은 **할 수 있는 일 자체를 좁혀 두는** 방식이라 어떤 SQL 이 와도 쓰기가 불가능합니다. 실무에서 조회 전용 계정을 따로 파는 것이 바로 이 발상입니다.

## 스키마 설명이 모델의 눈이다
좋은 스키마 설명에는 네 가지가 들어갑니다. **표와 열 이름**, **표끼리의 관계**, **각 열이 무슨 값을 갖는지**(예: `personal_data` 는 '예'/'아니오'), **지켜야 할 규칙**.

In [ ]:
# [제공 코드] 표 구조 설명 - 모델은 데이터베이스를 볼 수 없습니다. 이 글이 모델이 아는 전부입니다.
SCHEMA_PROMPT = """너는 사내 AI 서비스 도입 대장을 조회해 답하는 담당자다.
아래 표만 존재한다. 반드시 run_select 도구로 SELECT 를 실행해 확인한 값으로 답한다.

표 구조(sqlite):
  ai_service(service_id text, service_name text, vendor text, category text,
             personal_data text, overseas_transfer text, monthly_cost int, review_status text)
    -- 도입했거나 검토 중인 AI 서비스 목록.
    -- personal_data      : 그 서비스에 개인정보를 입력하게 되는가. '예' / '아니오'
    -- overseas_transfer  : 데이터가 국외 서버로 나가는가. '예' / '아니오'
    -- monthly_cost       : 1인당 월 이용료(원)
    -- review_status      : 도입 심사 상태. '승인' / '검토중' / '보류'
  ai_request(request_id int, service_id text, dept text, seats int,
             request_date text, status text)
    -- 부서가 올린 사용 신청 내역.
    -- seats        : 신청 좌석 수
    -- request_date : 접수일 'YYYY-MM-DD'
    -- status       : 신청 처리 상태. '접수' / '검토중' / '승인' / '반려' / '보류'
  ai_request.service_id 는 ai_service.service_id 를 가리킨다.

규칙:
- SELECT 한 문장만 만든다. 위에 적힌 표와 열만 쓰고, 없는 이름을 지어내지 않는다.
- 값이 정해진 열은 위에 적힌 한국어 값 그대로 비교한다('예'/'아니오' 를 true·1 로 바꾸지 않는다).
- 결과를 사람이 읽을 한국어 문장으로 짧게 정리해 답한다."""

# 모델이 무엇을 보고 SQL 을 만드는지 눈으로 확인합니다.
print(SCHEMA_PROMPT)

In [ ]:
# 스키마 설명을 system_prompt 로 주고 조회 도구 하나를 붙인 에이전트.
Q_TABLE = '국외로 데이터가 나가는 서비스가 몇 개야?'

table_agent = create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT)


def show_sql(result):
    """에이전트가 만든 SQL 과 최종 답을 함께 보여 준다."""
    for message in result['messages']:
        if isinstance(message, AIMessage):     # SQL 은 AIMessage 의 도구 호출에 담긴다
            for call in message.tool_calls:
                print('만든 SQL:', call['args']['sql'])
    print('답      :', result['messages'][-1].text)


show_sql(table_agent.invoke({'messages': Q_TABLE}))

In [ ]:
# 묶어서 세는 질문 - GROUP BY 가 필요합니다.
show_sql(table_agent.invoke({'messages': '부서별 신청 좌석 합계를 많은 순으로 알려줘.'}))

In [ ]:
# 두 표를 이어야 답이 나오는 질문 - 신청(ai_request)과 서비스(ai_service)를 JOIN 합니다.
show_sql(table_agent.invoke({'messages': '국외로 데이터가 나가는 서비스를 신청한 부서를 알려줘.'}))

> 세 질문의 SQL 을 나란히 보세요. `WHERE`·`COUNT` 에서 시작해 `GROUP BY`·`ORDER BY` 를 지나 마지막에는 **두 표를 `JOIN`** 했습니다. **우리가 준 것은 표 구조 설명 한 덩어리뿐입니다** - 어느 표를 이으라고 하지 않았는데도 `ai_request.service_id 는 ai_service.service_id 를 가리킨다` 는 한 줄에서 JOIN 조건을 세웁니다. **스키마 설명의 품질이 곧 답의 품질**인 이유입니다.

(실호출이라 만들어지는 SQL 의 표현은 실행할 때마다 조금씩 다를 수 있습니다. 별칭이나 대소문자가 달라도 뜻이 같으면 같은 답이 나옵니다.)

### 🖐️ 함께 따라하기: 스키마에 표를 하나 더 알려 주기

사내에는 표가 하나 더 있습니다 — **AI 서비스 사용 중 접수된 문의·사고 기록**입니다. 아래 제공 셀이 그 표를 만들어 둡니다. 여러분이 할 일은 **모델의 눈을 넓혀 주는 것**입니다.

1. 제공 셀을 실행해 `ai_incident` 표를 만들고 `run_query('select * from ai_incident')` 로 내용을 확인하세요.
2. `SCHEMA_PROMPT` 에 **이 표 설명을 덧붙인** 새 문자열 `SCHEMA_PROMPT_2` 를 만드세요 — 열 이름과 각 열이 갖는 값(`issue_type` 은 개인정보유출·오답·비용초과, `resolved` 는 '예'/'아니오'), 그리고 **`ai_incident.service_id` 가 `ai_service.service_id` 를 가리킨다**는 것을 반드시 적습니다.
3. `create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT_2)` 로 `incident_agent` 를 만들고 `'아직 해결되지 않은 문의를 서비스 이름과 함께 알려줘.'` 를 `invoke` 한 뒤 `show_sql` 로 확인하세요.

**확인 기준**: `JOIN` 이 들어간 SQL 이 만들어지고, 해결되지 않은 문의가 서비스 **이름**으로 나옵니다(표에는 id 만 있으므로 JOIN 없이는 이름이 나올 수 없습니다).

In [ ]:
# [제공 코드] 따라하기용 표 하나 더 - 실행만 하세요(쓰기는 보통 연결로 합니다).
_conn.executescript('''
drop table if exists ai_incident;
create table ai_incident (
    incident_id   int  primary key,
    service_id    text not null references ai_service(service_id),
    dept          text not null,
    incident_date text not null,
    issue_type    text not null,
    resolved      text not null
) strict;
insert into ai_incident values
 (1, 's1', '개발팀',     '2026-06-02', '개인정보유출', '아니오'),
 (2, 's3', '데이터팀',   '2026-06-05', '오답',         '예'),
 (3, 's4', '고객지원팀', '2026-06-09', '개인정보유출', '예'),
 (4, 's5', '경영지원팀', '2026-06-11', '비용초과',     '아니오'),
 (5, 's1', '마케팅팀',   '2026-06-15', '오답',         '아니오');
''')
print('ai_incident 표 준비 완료')

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) run_query 로 ai_incident 표를 확인한다
# 2) SCHEMA_PROMPT 에 ai_incident 설명을 덧붙인 SCHEMA_PROMPT_2 를 만든다
#    (열 이름·값 종류·ai_service 와의 관계를 반드시 적는다)
# 3) 그 프롬프트로 incident_agent 를 만들어 미해결 문의를 서비스 이름과 함께 묻고 show_sql 로 본다

### ✅ 바로 확인 퀴즈

**1.** 모델은 데이터베이스를 볼 수 있나요? 없다면 무엇을 보고 SQL 을 만드나요?

<details><summary>정답 보기</summary>

볼 수 없습니다. **우리가 프롬프트에 적어 준 스키마 설명이 모델이 아는 전부**입니다. 그래서 열 이름과 **표끼리의 관계**를 정확히 적어 줘야 하고, 설명이 부실하면 없는 열을 지어냅니다.

</details>

**2.** `select 1 from ai_service where 1=0 union select 1` 은 왜 문자열 검사를 통과하나요? 그래서 무엇이 진짜 방어선인가요?

<details><summary>정답 보기</summary>

`select` 로 시작하고 세미콜론이 없어 검사 기준을 만족하기 때문입니다. 문자열 검사는 "SELECT 인가" 만 봅니다. **진짜 방어선은 읽기 전용 연결**입니다 — 어떤 SQL 이 오든 쓰기 자체가 불가능합니다(`attempt to write a readonly database`).

</details>

---
# 4. 도구 넷을 한 창구에 붙이기

## 이제 모아 둡니다
지금까지 도구를 셋 만들었습니다. 여기에 **상태 도구** 하나를 더합니다 — 모델은 **오늘이 며칠인지도 모릅니다**(학습이 끝난 시점에 시간이 멈춰 있습니다).

| 도구 | 어디서 답을 가져오나 | 어떤 질문에 |
|---|---|---|
| `search_policy` | **문서**(비정형) — 안내서를 검색 | "학습에 쓰려면 무엇을 알려야 하나요?" |
| `run_select` | **표**(정형) — 데이터베이스 조회 | "국외이전 서비스가 몇 개야?" |
| `business_day_after` | **바깥**(외부 API) — 공휴일을 받아 계산 | "검토가 언제 끝나?" |
| `get_today` | **상태** — 지금 시점 | "오늘 며칠이야?" |

<img src="images/세_갈래_도구.png" width="900">

*문서·표·바깥 세상 — 답이 있는 곳이 다르면 도구도 달라야 합니다.*

In [ ]:
# 상태 도구 하나를 더해 넷으로 만듭니다 - 모델은 '오늘' 도 모릅니다.
@tool
def get_today() -> str:
    """오늘 날짜를 YYYY-MM-DD 형식으로 돌려준다."""
    return date.today().isoformat()


# 지금까지 만든 것을 한 리스트에 모아 둔다 - 이 리스트가 곧 에이전트가 고를 수 있는 행동의 전부다.
ALL_TOOLS = [search_policy, run_select, business_day_after, get_today]
print('도구 넷:', [t.name for t in ALL_TOOLS])

<img src="images/멀티툴_라우팅.png" width="900">

*질문 하나가 들어오면, 모델은 각 도구의 이름과 설명만 읽고 하나를 고릅니다.*

In [ ]:
# 네 도구를 모두 붙인 에이전트 - 질문마다 무엇을 골랐는지 봅니다.
# 네 질문의 답이 저마다 다른 곳에 있습니다 - 문서 / 표 / 바깥 세상 / 지금 시점.
Q_POLICY = '이용자가 챗봇에 입력한 대화를 학습에 쓰려면 무엇을 알려야 하나요?'
Q_TABLE = '국외로 데이터가 나가는 서비스가 몇 개야?'
Q_DAYS = '2026-05-08 에 접수한 건이 보안검토 5영업일이면 언제 끝나?'
Q_TODAY = '오늘 며칠이야?'

desk = create_agent(
    model, ALL_TOOLS,
    system_prompt=SCHEMA_PROMPT + '\n\n개인정보 규정·안내서 내용은 search_policy 로 찾아 답한다.')

for q in [Q_POLICY, Q_TABLE, Q_DAYS, Q_TODAY]:
    res = desk.invoke({'messages': q})
    print(f'{q[:30]:32} -> {tool_names(res)}')

> 네 질문이 **각각 다른 도구**로 갔습니다. 우리가 `if` 문을 하나도 쓰지 않았다는 점이 핵심입니다 - 분기를 코드로 짜는 대신 **도구 설명을 잘 써 두는 것**으로 라우팅이 됩니다.

## 한 번에 둘을 물으면?
성격이 다른 두 가지를 한 문장으로 물어봅니다 - **표에서 셀 것**과 **문서에서 찾을 것**입니다.

In [ ]:
# 한 질문이 서로 다른 두 소스를 요구하면 - 표와 문서를 함께 부릅니다.
Q_MIX = ('개인정보를 입력하게 되는 서비스가 몇 개고, '
         '학습데이터에 개인정보가 섞이면 어떻게 하라고 돼 있나요?')

res_mix = desk.invoke({'messages': Q_MIX})
print('불린 도구:', tool_names(res_mix))
# 도구는 둘인데 이 횟수가 늘지 않았다면 한 메시지에 두 호출을 함께 담았다는 뜻이다.
print('모델이 말한 횟수:', step_count(res_mix))
print()
print('답:', res_mix['messages'][-1].text)

> 도구가 **둘** 불렸습니다. 구조화된 `tool_calls` 는 **리스트**라 한 메시지에 여러 개를 그대로 담을 수 있습니다. 옛 방식(`행동:` 한 줄에서 도구 하나)으로는 표현하기 어려운 모양이지요.

## 앞 도구의 결과가 있어야 다음 도구를 부를 수 있다면
이번에는 조금 다른 질문입니다. **"개발팀이 가장 먼저 올린 신청은 5영업일이면 며칠에 끝나?"** - 접수일을 **표에서 먼저 알아내야** 영업일 계산을 시작할 수 있습니다. 한 번에 끝나지 않습니다.

In [ ]:
# 앞 도구의 결과가 있어야 다음 도구를 부를 수 있는 질문 - 루프가 두 바퀴 돕니다.
Q_CHAIN = '개발팀이 가장 먼저 올린 신청은 보안검토 5영업일이면 며칠에 끝나?'

res_chain = desk.invoke({'messages': Q_CHAIN})

# 부른 순서대로 찍어 봅니다 - 첫 도구의 결과가 둘째 도구의 인자로 들어가는지 보세요.
for message in res_chain['messages']:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print('행동:', call['name'], call['args'])
print()
print('답:', res_chain['messages'][-1].text)

<img src="images/다단계_궤적.png" width="900">

*첫 관찰(접수일)이 둘째 행동의 입력이 됩니다 - 세 단계가 두 번 반복됩니다.*

> **첫 관찰이 둘째 행동의 입력이 되었습니다.** `run_select` 가 돌려준 `2026-05-08` 을 모델이 읽고, 그것을 `business_day_after` 의 인자로 넘겼습니다. 우리는 그 순서를 코드로 정하지 않았습니다 - **질문이 정한 것**이고, 모델은 관찰을 보고 다음 행동을 골랐을 뿐입니다. 이것이 1절에서 그림으로 본 **고리가 한 바퀴 더 도는** 장면입니다.

### 🖐️ 함께 따라하기: 계산 도구를 하나 더 붙여 다섯으로

월 이용료를 견적 내는 도구를 만들어 붙여 보세요. 단가는 표에 있으므로 **모델이 먼저 표를 조회하고**, 그 값을 이 도구에 넘기게 됩니다(방금 본 다단계 연쇄와 같은 모양입니다).

1. `@tool` 로 **`estimate_cost(unit_price: int, seats: int) -> str`** 를 만드세요 — 단가와 좌석 수를 곱해 `'월 000원'` 처럼 돌려줍니다. docstring 에 *언제 쓰는 도구인지* 와 **단가는 표에서 조회해 넘겨야 한다**는 점을 적으세요.
2. `ALL_TOOLS + [estimate_cost]` 로 **`desk5`** 를 만듭니다(`system_prompt` 는 `desk` 와 같게).
3. `'고객상담 챗봇을 25명이 쓰면 월 얼마야?'` 를 `invoke` 하고, **불린 도구와 인자**를 순서대로 출력하세요.

**확인 기준**: `run_select` 가 먼저 불려 단가를 가져오고, 그다음 `estimate_cost` 가 그 단가와 `25` 를 인자로 받습니다. 도구를 늘려도 **앞의 도구들이 흔들리지 않는** 것이 좋은 설명의 증거입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) @tool 로 estimate_cost(unit_price: int, seats: int) -> str 를 만든다
#    docstring 에 언제 쓰는지와 '단가는 표에서 조회해 넘긴다' 를 적는다
# 2) ALL_TOOLS + [estimate_cost] 로 desk5 를 만든다 (system_prompt 는 desk 와 같게)
# 3) '고객상담 챗봇을 25명이 쓰면 월 얼마야?' 를 invoke 하고 불린 도구와 인자를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 도구가 넷인데 우리는 `if` 문을 하나도 쓰지 않았습니다. 분기는 어디서 일어나나요?

<details><summary>정답 보기</summary>

**모델 안에서** 일어납니다. 모델은 질문과 **각 도구의 이름·docstring** 을 보고 무엇을 부를지 정합니다. 그래서 라우팅 품질은 코드가 아니라 **설명의 품질**에 달려 있습니다.

</details>

**2.** 도구를 둘 부르는 두 경우 — "동시에 둘" 과 "앞의 결과로 다음" — 은 기록에서 어떻게 다른가요?

<details><summary>정답 보기</summary>

서로 무관한 둘이면 **한 `AIMessage` 의 `tool_calls` 에 나란히** 담겨 한 바퀴에 처리됩니다. 앞 결과가 있어야 다음을 부를 수 있으면 **`AIMessage` → `ToolMessage` → `AIMessage` → `ToolMessage`** 로 **모델이 말하는 횟수가 늘어납니다.** 그 횟수를 정하는 것은 우리가 아니라 **질문**입니다.

</details>

---
# 5. 도구 선택을 좌우하는 것과 틀어질 때의 처방

앞 절에서 우리는 `if` 한 줄 없이 도구가 갈렸다고 했습니다. 그러면 **무엇이** 그 선택을 정할까요? 이 절은 코드를 돌리지 않고 그 원리와 처방만 정리합니다. **오늘 배운 것 중 실무에서 가장 오래 붙잡게 될 내용**입니다.

## 선택을 정하는 것은 도구의 이름과 설명입니다
모델이 도구를 고를 때 보는 것은 딱 둘입니다 — **도구 이름**과 **docstring**(그리고 인자 이름·타입). 함수 본문은 보지 못합니다. 그래서 같은 질문·같은 모델이라도 설명이 다르면 선택이 달라집니다.

| | 두루뭉술한 설명 | 구체적인 설명 |
|---|---|---|
| 도구 이름 | `lookup1` · `lookup2` | `service_status` · `policy_clause` |
| docstring | "정보를 찾는다" · "데이터를 조회한다" | "AI 서비스 이름으로 도입 검토 상태(승인·검토중·보류)를 알려준다" |
| 같은 질문을 세 번 던지면 | 고르는 도구와 호출 횟수가 **실행마다 흔들린다** | **매번 같은 도구**를 한 번에 고른다 |

여기서 놓치면 안 되는 것이 있습니다. **`temperature=0` 이어도** 흔들립니다. 온도는 **표현의 다양성**을 줄일 뿐, **판단 근거를 만들어 주지 않습니다.** 근거는 오직 **우리가 쓴 설명**에서 나옵니다.

> **실행할 때마다 고르는 도구나 호출 횟수가 달라진다면, 모델이 변덕스러운 것이 아니라 설명이 부족하다는 신호입니다.** 도구를 만들 때 가장 오래 붙잡고 있어야 할 줄은 docstring 입니다.

## 증상 네 가지 — 무엇이 보이고, 어디를 보고, 무엇을 고치나

"설명이 나쁘다" 는 **진단**이고, 우리가 마주치는 것은 **증상**입니다. 증상은 넷이고 **원인도 처방도 각각 다릅니다.**

| 증상 | 기록의 어디에서 잡히나 | 처방 |
|---|---|---|
| **1) 안 부름** — 도구를 안 부르고 지어낸 답을 한다 | `tool_calls` 가 **비어 있다** | docstring 에 **언제 쓰는지**와 **받을 수 있는 값**을 적는다 |
| **2) 잘못 고름** — 엉뚱한 도구가 불린다 | `tool_calls` 의 **이름** | 이름·설명을 **좁힌다**(`check` → `policy_lookup`) |
| **3) 조용한 실패** — 맞게 불렸는데 결과가 "없음" | `tool_calls` 의 **인자**와 `ToolMessage` 의 **내용** | 도구가 **인자를 다듬고**, 없으면 `없음` 대신 **안내 문자열** |
| **4) 과다 호출** — 같은 도구를 반복해 부른다 | `tool_calls` 의 **개수** | 반복 유도 문구를 빼고 **결과가 확정임**을 밝힌다 |

> **세 번째가 특히 고약합니다.** 도구는 맞게 불렸고 에러도 나지 않았습니다. 담당자가 있는데 "없습니다" 라고 답하게 되지요. **에러가 안 났다고 맞은 것이 아닙니다** — 기록의 **인자**까지 봐야 잡힙니다. 없는 값에 `없음` 을 돌려주면 실패인지 진짜 없는 것인지 구분되지 않지만, 안내 문자열로 돌려주면 모델이 읽고 사용자에게 되묻거나 안내합니다.

## 네 증상이 그대로 보이는 도구
실행하지 않습니다 — **어디가 문제인지 읽어 보세요.**

```python
# _OWNER = {'s1': '보안팀 김서연', 's3': '보안팀 박도윤', 's4': '법무팀 이하준'}  <- 담당자 표

@tool
def check(query: str) -> str:
    """확인한다."""                      # ← 2) 이름·설명이 넓어 아무 질문에나 걸린다
    return '확인 완료'


@tool
def service_owner(service_id: str) -> str:
    """서비스 id 를 받아 그 서비스의 검토 담당자를 돌려준다."""   # ← 1) 받을 수 있는 값이 없다
    return _OWNER.get(service_id, '없음')   # ← 3) 받은 글자를 그대로 열쇠로 쓴다('S3' 면 없음)


@tool
def ping_vendor(vendor: str) -> str:
    """벤더 상태 페이지가 살아 있는지 확인한다.

    확실히 하려면 여러 번 불러 보는 것이 좋다."""   # ← 4) 반복을 시키는 문장
    return f'{vendor}: 정상'
```

## 좋은 docstring 점검표
도구를 하나 만들 때마다 이 넷을 확인하면 네 증상이 함께 막힙니다.

1. **무엇을 하는 도구인지** — 다른 도구와 겹치지 않게 좁게
2. **언제 쓰는지** — 어떤 질문이 오면 이 도구인지
3. **받을 수 있는 값** — id 목록·이름 형식·날짜 형식처럼 구체적으로
4. **결과의 모양**, 그리고 **한 번이면 충분하다는 것**

### ✅ 바로 확인 퀴즈

**1.** 같은 질문·같은 모델(`temperature=0`)인데 고르는 도구가 흔들립니다. 원인은?

<details><summary>정답 보기</summary>

**도구 설명이 판단 근거를 주지 못했기 때문**입니다. "정보를 찾는다"·"데이터를 조회한다" 로는 어느 쪽이 이 질문에 맞는지 알 수 없습니다. 온도는 **표현의 다양성**을 줄일 뿐 판단 근거를 만들어 주지 않습니다.

</details>

**2.** 도구가 **맞게 불렸는데** 답이 "없습니다" 로 나옵니다. 어디를 먼저 보나요?

<details><summary>정답 보기</summary>

**도구에 넘어간 인자**를 봅니다. 모델이 넘긴 표면형(`'S3'`)과 우리 데이터의 열쇠(`'s3'`)가 다를 수 있습니다. 처방은 **도구가 인자를 다듬는 것**이고, 그래도 없으면 **안내 문자열**을 돌려줍니다.

</details>

**3.** 같은 도구를 여러 번 반복해 부릅니다. docstring 에서 무엇을 볼까요?

<details><summary>정답 보기</summary>

**반복을 부추기는 문구**가 있는지 봅니다("확실히 하려면 여러 번" 같은). 그리고 결과가 애매하지 않게 **확정적인 문장**으로 돌려줍니다.

</details>

---
# 6. 저장소를 갈아 끼워도 도구는 그대로

## 이 절이 보이려는 것 하나
지금까지 이 노트북의 검색은 **노트북을 켤 때마다 다시 만든 색인**이었습니다. 커널을 끄면 사라집니다. 그런데 여러분은 **SQL 단원에서 이미 클라우드 저장소를 만들어 두었습니다** — Supabase(PostgreSQL) 위에 `pgvector` 를 얹고, 연구비 FAQ 를 임베딩해 넣고, `match_faq` 검색 함수까지 만들었지요.

| | SQL 단원에서 한 것 | 여기서 하는 것 |
|---|---|---|
| 검색 호출 | `supabase.rpc('match_faq', ...)` 를 직접 | **그대로** 부른다 |
| 결과 모양 | 딕셔너리 목록을 표로 본다 | **`Document` 목록**으로 바꾼다 |
| 그다음 | 사람이 읽는다 | **`@tool` 로 감싸** 에이전트에 붙인다 |

> **새로 배우는 것은 딱 하나입니다.** 저장소가 로컬이든 클라우드든 **부품 규약**(문자열 → `Document` 목록 → `@tool`)만 맞추면 지금까지 쓰던 에이전트에 **그대로 꽂힙니다.** 그 자료가 마침 다른 주제(연구비 FAQ)인 것은 SQL 단원의 산출물을 그대로 쓰기 때문입니다.

### 준비물
1. SQL 단원 폴더의 `.env` 에 있던 `SUPABASE_URL` · `SUPABASE_ANON_KEY` 를 **이 폴더의 `.env` 에도** 넣으세요.
2. 그 프로젝트에 `faq_docs` 표가 있고 **자료가 적재돼 있어야** 합니다.

> 이 절은 **접속을 전제로** 씁니다 — 준비가 안 된 상태로 실행하면 아래 첫 셀에서 바로 에러가 납니다(무엇이 빠졌는지 그 자리에서 드러나게 하려는 것입니다).

In [ ]:
# [제공 코드] 클라우드 벡터DB 연결 - SQL 단원에서 만든 Supabase 프로젝트를 그대로 씁니다(실행만 하세요).
from supabase import create_client

from langchain_core.documents import Document

# .env 에 SUPABASE_URL / SUPABASE_ANON_KEY 가 있어야 합니다 - SQL 단원 폴더의 .env 에서 옮겨 오세요.
project_url = os.environ['SUPABASE_URL']
anon_key = os.environ['SUPABASE_ANON_KEY']
supabase = create_client(project_url, anon_key)

# 연결만으로는 부족합니다 - 표에 자료가 있고 검색 함수까지 만들어졌는지 여기서 확인합니다.
loaded = supabase.table('faq_docs').select('faq_id').limit(1).execute()
print('faq_docs 적재 확인:', len(loaded.data), '건 이상')

# 0 으로 채운 벡터로 한 번 불러 검색 함수(match_faq)가 있는지 확인합니다.
supabase.rpc('match_faq', {'query_embedding': [0.0] * 768,
                           'match_count': 1, 'filter_category': None}).execute()
print(f'클라우드 벡터DB 연결 완료 - {project_url}')

In [ ]:
# 검색기의 규약에 맞춥니다 - 문자열을 넣으면 Document 목록이 나오게.
Q_CLOUD = '정산 방법과 사용실적보고서 제출은 어떻게 하나요?'

def cloud_faq_search(query, k=3, category=None):
    """클라우드 벡터DB 에서 질문과 가까운 FAQ 를 찾아 Document 목록으로 돌려준다."""
    # 질문을 벡터로 바꿔 SQL 단원에서 만든 검색 함수(match_faq)에 넘긴다.
    #   적재할 때와 같은 임베딩 모델이라야 순위가 맞는다.
    rows = supabase.rpc('match_faq', {'query_embedding': embeddings.embed_query(query),
                                      'match_count': k,
                                      'filter_category': category}).execute().data
    return [Document(page_content=f"질문: {r['question']}\n답변: {r['answer']}",
                     metadata={'faq_id': r['faq_id'], 'category': r['category'],
                               'team_name': r['team_name'],
                               'similarity': round(r['similarity'], 3)})
            for r in rows]


for d in cloud_faq_search(Q_CLOUD):
    print(d.metadata, '|', d.page_content.splitlines()[0][:40])


> 꼬리표에 `team_name` 이 붙어 있는 것을 보세요. `faq_docs` 표에는 없는 값입니다 - `match_faq` 함수 안의 `JOIN` 이 담당팀 표에서 붙여 온 것입니다. **저장소가 관계형 데이터베이스라서** 검색 결과에 관련 정보를 함께 실어 올 수 있습니다.

규약을 맞췄으니 남은 일은 2절에서 한 것과 똑같습니다.

In [ ]:
# 규약을 맞췄으니 감싸기만 하면 됩니다 - 2절의 search_policy 와 같은 모양입니다.
Q_CLOUD = '정산 방법과 사용실적보고서 제출은 어떻게 하나요?'

@tool
def search_research_faq(query: str) -> str:
    """연구비 관리 시스템 FAQ 에서 질문과 관련된 안내를 찾아 돌려준다.

    연구비 집행·정산·카드 등록처럼 연구비 규정이나 시스템 사용법을 물을 때 쓴다.
    """
    return '\n\n'.join(d.page_content for d in cloud_faq_search(query, k=3))


cloud_agent = create_agent(model, [search_research_faq])
res_cloud = cloud_agent.invoke({'messages': Q_CLOUD})
print('불린 도구:', tool_names(res_cloud))
print('답      :', res_cloud['messages'][-1].text[:200])


> **바뀐 것은 도구 안쪽 한 줄뿐입니다.** 로컬 색인을 부르던 자리에 클라우드 검색을 넣었을 뿐, `@tool` 도 `create_agent` 도 기록을 읽는 방법도 그대로입니다.

## 조건으로 좁히고, 의미로 정렬하기
`match_faq` 에는 인자가 하나 더 있습니다 - `filter_category`. 값을 주면 **그 분류 안에서만** 찾습니다. 함수 안에서 `WHERE` 로 먼저 거르고 남은 것만 벡터로 줄 세우는 것이지요.

In [ ]:
# 같은 질문을 조건 없이 / 분류로 좁혀서 - 돌아온 문서의 분류를 견줍니다.
Q_CLOUD_FILTER = '카드는 어떻게 등록하나요?'
CAT_FILTER = '환경설정'

# 같은 질문·같은 k 인데 category 인자 하나로 결과가 달라진다.
print('조건 없이:', [d.metadata['category'] for d in cloud_faq_search(Q_CLOUD_FILTER, k=3)])
print('분류로 좁혀:', [d.metadata['category']
                   for d in cloud_faq_search(Q_CLOUD_FILTER, k=3, category=CAT_FILTER)])


### 로컬 Chroma 와 클라우드 pgvector 중 무엇을 고르나

| | 로컬 벡터DB (Chroma) | 클라우드 pgvector (Supabase) |
|---|---|---|
| 시작하기 | 설치만 하면 끝, 가입 없음 | 프로젝트 생성·키 발급이 필요 |
| 자료가 남는가 | 커널을 끄면 사라진다(이 노트북 방식) | **남는다** - 한 번 넣으면 계속 |
| 여러 사람이 | 각자 자기 컴퓨터에 따로 | **한 곳을 함께** 본다 |
| 조건 + 의미 | 메타데이터 필터로 가능하지만 조합이 늘면 번거롭다 | **`WHERE` 한 줄** |
| 다른 표와 잇기 | 따로 관리해야 한다 | **`JOIN`** 으로 함께 실어 온다 |
| 속도·비용 | 내 컴퓨터, 네트워크 없음 | 호출마다 네트워크를 탄다 |

**고르는 기준은 이렇습니다.** 수업·프로토타입처럼 **혼자, 잠깐** 쓰는 것이라면 로컬이 편합니다. 여러 사람이 같은 자료를 보고, 자료가 계속 쌓이고, **조건과 의미를 함께 걸어야** 한다면 클라우드 쪽이 맞습니다. "최신이라 좋다" 가 아니라 **무엇이 필요한가**로 고르세요.

### 🖐️ 함께 따라하기: 분류를 고정한 창구 만들기

데모는 분류를 **모델이 고르게** 두었습니다. 이번에는 **정산 담당 창구**처럼 분류가 이미 정해진 도구를 만들어 보세요.

1. `cloud_faq_search` 로 `'연구비정산'` 분류 안에서 `'정산 서류는 어디에 등록하나요?'` 를 `k=2` 로 검색해, 각 결과의 **`metadata['category']`** 와 **`metadata['similarity']`** 를 출력하세요.
2. 그 검색을 `@tool` 로 감싼 **`search_settlement_faq(query: str) -> str`** 를 만드세요 — 분류는 도구 안에 `'연구비정산'` 으로 **고정**하고(모델이 정하지 않습니다), 찾은 내용을 빈 줄로 이어 돌려줍니다. docstring 에 *언제 쓰는 도구인지* 를 적으세요.
3. `create_agent(model, [search_settlement_faq])` 로 에이전트를 만들어 같은 질문을 `invoke` 하고, `tool_names` 와 최종 답을 출력하세요.

**확인 기준**: 1번에서 출력된 분류가 **둘 다 `연구비정산`** 입니다(조건이 먼저 걸리기 때문입니다). **창구가 정해져 있으면 모델에게 고르게 할 이유가 없다** — 그것이 이 연습의 요점입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) cloud_faq_search 로 '연구비정산' 분류에서 '정산 서류는 어디에 등록하나요?' 를 k=2 로 검색하고
#    각 결과의 metadata['category'] 와 metadata['similarity'] 를 출력한다
# 2) 그 검색을 @tool 로 감싼 search_settlement_faq(query) 를 만든다 (분류는 '연구비정산' 고정)
# 3) create_agent 로 붙여 같은 질문을 invoke 하고 tool_names 와 최종 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 클라우드 저장소를 쓰려고 새로 배운 것은 무엇인가요?

<details><summary>정답 보기</summary>

**검색 결과를 `Document` 목록으로 바꾸는 것 하나**입니다. 그 규약만 맞추면 `@tool`·`create_agent`·출처 표기는 **앞 절들과 똑같이** 동작합니다.

</details>

**2.** 적재할 때와 검색할 때 임베딩 모델이 다르면 어떻게 되나요?

<details><summary>정답 보기</summary>

**에러 없이 순위만 틀어집니다.** 차원만 맞으면 계산은 되기 때문입니다 — 5절에서 본 **조용한 실패**의 또 다른 얼굴입니다. 그래서 적재·검색이 같은 모델을 쓰는지 늘 확인해야 합니다.

</details>

---
## 이번 강의 정리

| 개념 | 핵심 | 코드 |
|---|---|---|
| ReAct 세 단계 | 생각 → 행동 → 관찰의 **반복** | `result['messages']` 가 그 기록 |
| 메시지 기록 ↔ 세 단계 | `AIMessage`(행동) · `ToolMessage`(관찰) | `tool_names` · `step_count` |
| 단계 관찰 | 한 단계가 끝날 때마다 쌓인 상태를 본다 | `agent.stream(..., stream_mode='values')` |
| 외부 API 도구 | 실패를 **문자열로** 돌려준다 | `try` / `except requests.RequestException` |
| 문서 도구 | 작은 조각으로 **찾고**, 앞뒤까지 붙여 **넘긴다** | `with_neighbors(store, hit, window=1)` |
| 출처 | 꼬리표에 쪽·소제목을 담아 두면 답에 붙일 수 있다 | `metadata` |
| 체인 vs 에이전트 | 정해진 흐름이면 체인, 무엇이 올지 모르면 에이전트 | — |
| 표 도구 | 스키마 설명이 **모델의 눈** | `create_agent(model, [run_select], system_prompt=...)` |
| 두 겹 가드 | 문자열 검사 + **읽기 전용 연결** | `mode=ro` 연결이 진짜 방어선 |
| 멀티툴 라우팅 | `if` 없이 **설명으로** 분기한다 | 도구의 이름과 docstring |
| 다단계 연쇄 | 앞 관찰이 다음 행동의 입력이 된다 | 반복 횟수를 정하는 것은 **질문** |
| 증상 네 가지 | 안 부름 · 잘못 고름 · **조용한 실패** · 과다 호출 | 기록의 **인자**까지 본다 |
| 근거 기반 답변 | 찾은 내용에만, 없으면 모른다고 | `system_prompt` 로 못박기 |

- **답이 있는 곳이 다르면 도구도 달라야 합니다** — 문서·표·바깥 세상. 검색 하나로 다 덮을 수 없습니다.
- **라우팅 품질은 코드가 아니라 설명의 품질입니다.** 도구를 늘릴수록 이 말이 무거워집니다.
- **에러가 안 난다고 맞은 것이 아닙니다.** 조용한 실패를 잡으려면 기록의 **인자**까지 봐야 합니다.

## ⏭️ 예고

오늘 우리는 검색을 **만들고 도구로 붙였습니다.** 그런데 한 번도 묻지 않은 것이 있습니다 — **그 검색은 얼마나 맞히고 있나요?** "그럴듯해 보인다" 는 근거가 아닙니다.

다음 시간에는 오늘 만든 검색을 **숫자로 잽니다.** 눈금 네 개를 손으로 만들고, 그 숫자를 **어떻게 읽어야 하는지**까지 배웁니다 — **재지 못하면 고칠 수도 없기 때문입니다.**

수고하셨습니다!